[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module1/04-control-flow.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module1/04-control-flow.ipynb)

# Lesson 4 — Control Flow

**Module 1 — Python Fundamentals** | ⏱ 20 min

Control flow determines the order in which statements execute. Without it, programs would always run straight from top to bottom. By using conditional statements, you can make your program make decisions — executing different code depending on the state of your data. Python offers `if/elif/else`, ternary expressions, and the modern `match/case` statement introduced in Python 3.10.

## Learning Objectives
- Write `if/elif/else` chains to handle multiple conditions
- Apply the guard clause pattern for cleaner code
- Use ternary expressions for concise conditional assignment
- Use `match/case` (Python 3.10+) for structural pattern matching
- Understand nested conditions and when to avoid them

## if / elif / else

The `if` statement is the foundation of decision-making in Python. If the condition is `True`, the indented block runs; otherwise Python moves to the `elif` (else-if) or `else` clauses. Python uses **indentation** (4 spaces by convention) rather than braces `{}` to define code blocks — this is non-negotiable. Any expression that evaluates to a truthy or falsy value can be used as a condition.

In [ ]:
# Basic if/elif/else — grading example
def letter_grade(score):
    """Return a letter grade for a numeric score."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    else:
        return "F"

# Test several scores
test_scores = [95, 83, 72, 61, 45]
for score in test_scores:
    print(f"Score {score}: Grade {letter_grade(score)}")

In [ ]:
# Truthy and falsy values — Python considers these as False:
# False, None, 0, 0.0, "", [], {}, set()

def check_value(value):
    if value:  # No need to write 'if value != 0' or 'if value != ""'
        print(f"'{value}' is TRUTHY")
    else:
        print(f"'{value}' is FALSY")

check_value(42)
check_value(0)
check_value("hello")
check_value("")
check_value([1, 2, 3])
check_value([])       # Empty list is falsy
check_value(None)

## Guard Clauses — Early Returns for Cleaner Code

A **guard clause** is an early `return` (or `raise`) at the top of a function that handles edge cases before the main logic. This pattern — sometimes called "fail fast" or "early exit" — avoids deeply nested code. Instead of wrapping your main logic in a big `if` block, you handle the exceptional cases first and return early. The happy path code then runs without indentation, making it easier to read.

In [ ]:
# WITHOUT guard clauses — deeply nested, hard to follow
def process_order_bad(user, order, inventory):
    if user is not None:
        if user.get('is_active'):
            if order.get('quantity', 0) > 0:
                if inventory.get(order['item'], 0) >= order['quantity']:
                    # Finally — the actual logic
                    return f"Order placed for {order['quantity']} x {order['item']}"
                else:
                    return "Out of stock"
            else:
                return "Invalid quantity"
        else:
            return "User account is inactive"
    else:
        return "No user provided"

# WITH guard clauses — flat, easy to scan
def process_order(user, order, inventory):
    if user is None:
        return "No user provided"
    if not user.get('is_active'):
        return "User account is inactive"
    if order.get('quantity', 0) <= 0:
        return "Invalid quantity"
    if inventory.get(order['item'], 0) < order['quantity']:
        return "Out of stock"

    # Happy path — runs only when all guards pass
    return f"Order placed for {order['quantity']} x {order['item']}"

# Test it
active_user = {'name': 'alice', 'is_active': True}
stock = {'headphones': 10, 'keyboard': 3}
order = {'item': 'headphones', 'quantity': 2}

print(process_order(active_user, order, stock))
print(process_order(None, order, stock))
print(process_order(active_user, {'item': 'headphones', 'quantity': 20}, stock))

## Ternary Expressions

A **ternary expression** (also called a conditional expression) condenses a simple `if/else` assignment into a single line: `value_if_true if condition else value_if_false`. This is ideal when you are choosing between two values based on a condition. It should only be used for simple cases — if the logic is complex, a regular `if/else` block is clearer. Ternary expressions are frequently used in list comprehensions and function arguments.

In [ ]:
# Ternary operator syntax: value_if_true if condition else value_if_false

age = 20

# Long form with if/else
if age >= 18:
    status = "adult"
else:
    status = "minor"
print(status)  # adult

# Equivalent ternary expression
status = "adult" if age >= 18 else "minor"
print(status)  # adult

# Practical examples
temperature = 28
warning = "Hot" if temperature > 35 else "OK"
print(f"Temperature status: {warning}")

# Ternary in f-strings
items_in_cart = 3
print(f"You have {items_in_cart} item{'s' if items_in_cart != 1 else ''} in your cart.")
# "You have 3 items in your cart."

# Ternary for clamping a value to a range (nested ternary — use sparingly)
raw_score = 105
clamped = 100 if raw_score > 100 else (0 if raw_score < 0 else raw_score)
print(f"Clamped score: {clamped}")  # 100

## match / case — Structural Pattern Matching (Python 3.10+)

Python 3.10 introduced `match/case`, which goes far beyond a simple switch statement. It supports **structural pattern matching** — matching against patterns including literals, sequences, mappings, class instances, and more. The `_` wildcard matches anything (like `default` in other languages). Unlike a chain of `if/elif`, `match/case` can destructure data while matching, making it especially powerful for processing structured data like API responses.

In [ ]:
import sys
print(f"Python version: {sys.version_info.major}.{sys.version_info.minor}")
# match/case requires Python 3.10 or later — Colab uses 3.10+

# Basic match/case — matching literal values
def http_status_message(status_code):
    match status_code:
        case 200:
            return "OK"
        case 201:
            return "Created"
        case 400:
            return "Bad Request"
        case 401:
            return "Unauthorized"
        case 404:
            return "Not Found"
        case 500:
            return "Internal Server Error"
        case _:  # Wildcard — matches anything not caught above
            return f"Unknown status code: {status_code}"

for code in [200, 404, 500, 418]:
    print(f"{code}: {http_status_message(code)}")

In [ ]:
# Advanced match/case — matching sequences and extracting values

def process_command(command):
    """Process a command given as a list of strings."""
    match command:
        case ["quit"]:
            return "Quitting application."
        case ["go", direction]:
            return f"Moving {direction}."
        case ["go", direction, speed]:
            return f"Moving {direction} at {speed} speed."
        case ["pick", "up", item]:
            return f"Picked up {item}."
        case ["drop", *items]:  # *items captures remaining elements
            return f"Dropped: {', '.join(items)}."
        case _:
            return f"Unknown command: {command}"

commands = [
    ["go", "north"],
    ["go", "south", "fast"],
    ["pick", "up", "sword"],
    ["drop", "shield", "potion"],
    ["quit"],
    ["fly"]
]
for cmd in commands:
    print(process_command(cmd))

In [ ]:
# match/case with dictionaries and guard clauses (when)

def route_api_request(request):
    """Route an API request based on method and path."""
    match request:
        case {"method": "GET", "path": path} if path.startswith("/api/"):
            return f"Fetching API resource: {path}"
        case {"method": "POST", "path": path, "body": body}:
            return f"Creating resource at {path} with data: {body}"
        case {"method": "DELETE", "path": path}:
            return f"Deleting resource: {path}"
        case {"method": method}:
            return f"Method {method} not handled."
        case _:
            return "Invalid request format."

requests = [
    {"method": "GET", "path": "/api/users"},
    {"method": "POST", "path": "/api/items", "body": {"name": "widget"}},
    {"method": "DELETE", "path": "/api/items/42"},
    {"method": "PATCH"},
]
for req in requests:
    print(route_api_request(req))

## Nested Conditions and Anti-Patterns

Nesting `if` statements inside other `if` statements is sometimes necessary, but deep nesting quickly makes code hard to read and maintain. The rule of thumb is to keep nesting to two levels at most. Common anti-patterns include unnecessary `else` after a `return`, comparing booleans to `True/False` explicitly, and writing conditions that are always true or always false. Recognising these patterns helps you write cleaner code.

In [ ]:
# ANTI-PATTERNS to avoid

# 1. Don't compare booleans explicitly
is_valid = True

# Bad:
if is_valid == True:
    print("valid (bad style)")

# Good:
if is_valid:
    print("valid (good style)")

# 2. Don't use 'else' after a 'return'
def get_discount_bad(is_member):
    if is_member:
        return 0.15
    else:          # This else is redundant — if we returned above, we never reach here
        return 0.0

def get_discount_good(is_member):
    if is_member:
        return 0.15
    return 0.0     # The else is implicit

# 3. Avoid deeply nested conditions — use guard clauses or combine with 'and'/'or'
def check_access_bad(user, resource):
    if user:
        if user.get('active'):
            if resource:
                if resource.get('public') or user.get('admin'):
                    return True
    return False

def check_access_good(user, resource):
    if not user or not user.get('active'):
        return False
    if not resource:
        return False
    return resource.get('public') or user.get('admin', False)

test_user = {'active': True, 'admin': False}
test_resource = {'public': True}
print(check_access_good(test_user, test_resource))  # True

## Practice Exercises

1. Write a function `bmi_category(weight_kg, height_m)` that calculates BMI (`weight / height**2`) and returns a string: `"Underweight"` (BMI < 18.5), `"Normal"` (18.5-24.9), `"Overweight"` (25-29.9), or `"Obese"` (30+). Use guard clauses to reject invalid inputs (negative or zero values).
2. Rewrite this nested condition using guard clauses: `if user: if user['balance'] > 0: if amount <= user['balance']: return True return False`.
3. Use a ternary expression to write a one-liner that returns `"fizz"` if a number is divisible by 3, `"buzz"` if divisible by 5, `"fizzbuzz"` if divisible by both, and the number itself otherwise.
4. Write a `match/case` statement that handles a shopping cart command: `("add", item, qty)`, `("remove", item)`, `("clear",)`, and `("checkout", discount_code)` — print an appropriate message for each.